# **Model: LiquidAI/LFM2.5-1.2B-Instruct**

In [1]:
# Install modules
!pip install transformers
!pip install datasets
!pip install peft
!pip install torch

In [2]:
import torch
from datasets import Dataset, load_dataset
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    TrainingArguments,
    Trainer,
    DataCollatorForLanguageModeling
)
from peft import LoraConfig, get_peft_model
from typing import Dict

In [3]:
# Model name and Json File name

MODEL_NAME = "LiquidAI/LFM2.5-1.2B-Instruct"
JSON_FILE = "/content/generated_primerobotics.json"
OUTPUT_DIR = './primerobotics'
SAVED_MODEL_DIR = './prime_robotics_lora'

In [4]:
# Get Tokenizer

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
tokenizer.pad_token = tokenizer.eos_token

config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/434 [00:00<?, ?B/s]

chat_template.jinja: 0.00B [00:00, ?B/s]

In [5]:
# Load the json file and inspect

dataset = load_dataset(
    'json',
    data_files=JSON_FILE,
    split='train'
)

print(dataset[0])

Generating train split: 0 examples [00:00, ? examples/s]

{'question': 'What are the four steps to become certified?', 'answer': 'Enroll in the program, complete hands-on training, finish the capstone project, and receive your certification with hiring support.'}


In [6]:
# Format the model to suit the model's chat template

def build_chat(example: Dict) -> Dict:
  messages = [
      {'role': 'user', 'content': example['question']},
      {'role': 'assistant', 'content': example['answer']}
  ]
  text =  tokenizer.apply_chat_template(
      messages,
      add_generation_prompt=False,
      tokenize=False
  )
  return {'text': text}

In [7]:
# Apply the template to every row of dataset

dataset = dataset.map(build_chat)

Map:   0%|          | 0/382 [00:00<?, ? examples/s]

In [8]:
# Inspect the dataset

dataset[0]

{'question': 'What are the four steps to become certified?',
 'answer': 'Enroll in the program, complete hands-on training, finish the capstone project, and receive your certification with hiring support.',
 'text': '<|startoftext|><|im_start|>user\nWhat are the four steps to become certified?<|im_end|>\n<|im_start|>assistant\nEnroll in the program, complete hands-on training, finish the capstone project, and receive your certification with hiring support.<|im_end|>\n'}

In [9]:
# tokenize only the text(in model format) feature of dataset

def tokenize(batch):
  return tokenizer(
      batch['text'],
      truncation=True
  )

In [10]:
# apply tokenization to the whole dataset and remove example, answer and text features

dataset = dataset.map(
    tokenize,
    batched=True,
    remove_columns=['question', 'text', 'answer'])
dataset.set_format('torch')

Map:   0%|          | 0/382 [00:00<?, ? examples/s]

In [11]:
# Inpect the final dataset

dataset[0]

{'input_ids': tensor([    1,     1,     6,  6423,   708,  3493,   938,   779,  2759,  7441,
           811,  3117, 26035,   540,     7,   708,     6, 64015,   708,  2606,
          2753,   797,   779,  2209,   521,  5156,  6108, 10351,  4611,   521,
         14097,   779,  1906, 11108,  2826,   521,   810,  6984,  1180, 28360,
           916, 42341,  2258,   523,     7,   708]),
 'attention_mask': tensor([1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
         1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1])}

In [12]:
# Load the model weight, configs etc

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
    # device_map='auto'
)

model.safetensors:   0%|          | 0.00/2.34G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

In [13]:
# Configurations for Lora Adapter

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=['q_proj', 'v_proj'],
    lora_dropout=0.05,
    bias='none',
    task_type='CAUSAL_LM'
)

model = get_peft_model(model, lora_config)

In [14]:
# Inspect trainable parameters

model.print_trainable_parameters()

trainable params: 638,976 || all params: 1,170,979,584 || trainable%: 0.0546


In [15]:
# Set configurations/arguments for training args

training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=4,
    num_train_epochs=5,
    learning_rate=9e-4,
    logging_steps=10,
    save_strategy='epoch',
)

In [16]:
# Set data collater config

data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=False
)

In [17]:
# Set trainer config

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=dataset,
    data_collator=data_collator
)

In [18]:
# Train the model

trainer.train()

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:668: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


Step,Training Loss
10,3.553465
20,2.336219
30,1.717566
40,1.321107
50,1.114237
60,0.966891
70,0.877386
80,0.806359
90,0.703914
100,0.657825


/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:668: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)
/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:668: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)
/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:668: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)
/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:668: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


TrainOutput(global_step=120, training_loss=1.270983390013377, metrics={'train_runtime': 8513.0208, 'train_samples_per_second': 0.224, 'train_steps_per_second': 0.014, 'total_flos': 580777403541504.0, 'train_loss': 1.270983390013377, 'epoch': 5.0})

In [19]:
# Save the fine-tuned model and tokenizer

model.save_pretrained(SAVED_MODEL_DIR)
tokenizer.save_pretrained(SAVED_MODEL_DIR)

('./prime_robotics_lora/tokenizer_config.json',
 './prime_robotics_lora/chat_template.jinja',
 './prime_robotics_lora/tokenizer.json')

In [20]:
# Test the fined-tune model

def chatbot(prompt):
  messages = [{'role': 'user', 'content': prompt}]

  inputs = tokenizer.apply_chat_template(
      messages,
      add_generation_prompt=True,
      return_tensors='pt'
  ).to(model.device)

  with torch.no_grad():
    output = model.generate(
        **inputs,
        max_new_tokens=50,
        temperature=0.5,
        do_sample=True,
        eos_token_id=tokenizer.eos_token_id,
        pad_token_id=tokenizer.eos_token_id
    )
  response = tokenizer.decode(
      output[0],
      skip_special_tokens=True
  )
  return response.split('\nassistant\n')[-1].strip()

In [21]:

while True:
  print('\nAsk your question')
  prompt = input('User:   ')
  if not prompt.lower().strip() or prompt.lower().strip() in ['quit', 'exit']:
    break
  print(f'Bot:   {chatbot(prompt)}')


Ask your question
User:   what courses does prime robotics offer
Bot:   Prime Robotics offers courses in Robotics Development, AI & Machine Learning, Cloud Engineering, Cyber Security, and Software Development.

Ask your question
User:   what is the duration of cloud engineering
Bot:   The duration of the Cloud Engineering program is 6 months.

Ask your question
User:   what is the cost of cyber security
Bot:   The cost of a Cyber Security program is ₦350,000.

Would you like to enroll in the Cyber Security program?

Ask your question
User:   what does cyber security program entail
Bot:   It involves learning to identify and neutralize cyber threats, protecting organizational data, and ensuring secure digital operations.

Ask your question
User:   does prime robotics teach javascript
Bot:   Yes, both the Web Development track and the AI track include JavaScript as a core technology.

Ask your question
User:   does prime robotics teach media management
Bot:   Yes, the program includes 

KeyboardInterrupt: Interrupted by user

In [22]:
!zip -r ./prime_robotics_lora.zip /content/prime_robotics_lora

  adding: content/prime_robotics_lora/ (stored 0%)
  adding: content/prime_robotics_lora/README.md (deflated 65%)
  adding: content/prime_robotics_lora/chat_template.jinja (deflated 73%)
  adding: content/prime_robotics_lora/adapter_config.json (deflated 56%)
  adding: content/prime_robotics_lora/tokenizer_config.json (deflated 51%)
  adding: content/prime_robotics_lora/adapter_model.safetensors (deflated 8%)
  adding: content/prime_robotics_lora/tokenizer.json (deflated 81%)
